# CR-4 段A 揺れなかった単語の一覧の固定と探索（API なし・CPU）

凍結した計画 `notes/cr4_plan.md`（第3版、21a0b63）の段A です。**API には何も送りません。**

- CR-2・CR-3 の記録をすべて照合し、CR-3 の 264 語の量を作り直して CR-3 の結果と一致することを確かめます。
- CR-2 で一度も揺れなかった 340 語と、段B1 で新しく引く 262 個の切れ目の一覧を作り、予算を確かめます。
- あわせて②の探索（何が「答えの変わりやすさ」とつながっているか）を計算します。
- 記録は数とハッシュだけです。

**使い方**：CPU のランタイム。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で 5〜15 分。終わったら Claude に教えてね（Claude が段A の記録を Git に固定します）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-4 段A 揺れなかった単語の一覧の固定と探索（API なし・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-4 stage A lock', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'CR4_STAGEA'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "3feed77368cb7ff41c83ff75f3f03873b97252a5"
ANCESTORS = ["311e3008211ebaca0735abccf3501e6fcb8da3df", "da200a4805bdd727f4bb4b4094a08aa070c31a47", "c2659a50138f5983f2528f42d711e257c94bf579", "a49135f89441afe922d3a058ed0994559ebf8c79", "9711c5673c8d7eace445235aa00e2946d99cd443"]
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["21a0b63"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
from pork import cr3 as X
from pork import cr4 as Z
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

In [ ]:
# Cell 2 — CR-4 段A: CR-2・CR-3 の照合、NW、段B1 の一覧、予算の確かめ、②の探索（API なし）。数とハッシュだけ。
t0 = time.time()

G1_HEAD, G2S_HEAD, G2C_HEAD = "311e3008211ebaca0735abccf3501e6fcb8da3df", "da200a4805bdd727f4bb4b4094a08aa070c31a47", "c2659a50138f5983f2528f42d711e257c94bf579"
CANDIDATES_SHA = "c4bffbfa9632ed51a1222fae0bc65b256b3db0ae7e1b0d12e8bf70ac649f2339"
FILE_SHA = "731f2db9a590ac6c49ab7529d08dee0fcb09cc832a6462248e6939919319222c"
FINAL_ITEMS_SHA = "9ffd511cff0024e5106e50d18b5559f179485d10d528c973a2f458648ab8fee9"
STAGE3B_SHA = "1a70cc32b83dab56257911fa3c1ad985b30d6dbc64448907e2026ecd82b02343"
G2C_SUMMARY_SHA = "63c3654f4ec0dab2169db8ad67b2f781bb9e10ec81f390b83b36ae73f1d8faa8"
G3_RESULT_SHA = "2834ba20a1d2edd813d7a7fa0e7cf2084e17aa0c1f4e1e3aaa40a0043b6ce3c7"
CR2_CODE = ["pork/cr1_analysis.py", "pork/cr2.py", "pork/cr2_search.py", "pork/cr2_g2.py", "pork/cr2_g3.py", "pork/cr2_stage3b.py", "pork/cr2_run.py", "pork/cr1_run.py", "pork/alt_prelook.py"]
assert subprocess.run(["git", "diff", "--quiet", G1_HEAD, head, "--"] + CR2_CODE).returncode == 0, \
    "CR-2 の分析のコードが CR-2 のときと違います。止めます。"

cand, cands, prov, readme, pre = rebuild_stage2(FILE_SHA, CANDIDATES_SHA)
P_ = RN.paths()
ck = json.loads((Path("/content/drive/MyDrive") / P_["checkpoint"]).read_text())
assert ck["header"] == RN.header(G1_HEAD, cand["candidates_sha256"], CANDIDATES_SHA, RN.SDK_VERSION, C.DATASET_REVISION, FILE_SHA, FILE_SHA)
RN.validate(ck, cands)
g0 = json.loads((D / "G0/CR2_G0.json").read_text())
assert g0["pass"] and g0["code_head"] == G1_HEAD
g1 = RN.summarize_g1(ck, cands, g0["spent_usd"])
assert g1["verdict"] == "GO" and g1["final_items_sha256"] == FINAL_ITEMS_SHA, "G1 の結果が段3 の固定と違います。止めます。"
questions = {C.cid(c["id"]): c["question"] for c in cands}
ITEMS = RN.manifest_items(g1, ck, questions)
PRIOR = g0["spent_usd"] + g1["g1_spent_usd"]
PER_G1B = g1["G1b"]["estimate"]["per_request_usd_base"]      # early_stop applies the x1.2
del questions, cands

OUTC = D / "G2C"
REC2 = json.loads((REPO / "notes/cr2_lock_stage3b.json").read_text())
assert T3.stage3b_sha256(REC2) == STAGE3B_SHA, "Git の段3b の記録が変わっています。止めます。"
sc = OUTC / "CR2_G2C_summary.json"
assert hashlib.sha256(sc.read_bytes()).hexdigest() == G2C_SUMMARY_SHA, "G2c の記録が変わっています。止めます。"
assert hashlib.sha256((D / "G3/CR2_G3_result.json").read_bytes()).hexdigest() == G3_RESULT_SHA, "G3 の結果が変わっています。止めます。"
STC = json.loads((OUTC / "CR2_G2C_state.json").read_text())
assert STC["header"]["code_head"] == G2C_HEAD
REQC = G.Requests(ITEMS, T3.confirmation_items(REC2), "c", REC2["confirmation_manifest_sha256"])
G.verify_collected(STC, REQC, OUTC)
CONF2 = G.per_item_results(STC, REQC, OUTC)
rec2 = {int(c["id"]): c for c in REC2["items"]}
TEXTS = {C.cid(f["id"]): ck["items"][C.cid(f["id"])]["text"] for f in g1["final_items"]}
items2 = [C.build_sitem(i, f["id"], TEXTS[C.cid(f["id"])], rec2[int(f["id"])]["words"], tuple(rec2[int(f["id"])]["window"]),
                        CONF2.get(int(f["id"]), {}), depth=int(f["depth"]), seed=str(f["seed_sha256"]))
          for i, f in enumerate(g1["final_items"])]
TARGETS = X.build_targets(items2, g1["final_items"], TEXTS, CONF2)
CNT = X.counts(TARGETS)
MAXTOK = max(int(v) for v in STC["tokens"].values())
BUDGET = X.budget_check(CNT["cells"], MAXTOK)
assert BUDGET["K_F"] is not None, "段A の予算の確かめを K_F = 32 でも通りません。止めます（計画を直します）。"
K_F = BUDGET["K_F"]
ROWS, MSHA = X.manifest_rows3(ITEMS, TARGETS, K_F)
CR2_HASHES = {"g2c_summary_sha256": G2C_SUMMARY_SHA, "stage3b_sha256": STAGE3B_SHA, "g3_result_sha256": G3_RESULT_SHA}
LOCK = X.lock_record(TARGETS, FINAL_ITEMS_SHA, BUDGET, MSHA, len(ROWS), CR2_HASHES)
D3 = Path("/content/drive/MyDrive/DECISION_POINTS/cr3")

CR3_LOCK_SHA = "7a20da69b9453d80258d2b999ee9a3554fefca5201c27563bb0ff47d945d0247"
CR3_STAGEB_SUMMARY_SHA = "0ee591cf4573e00a5346502b49d1c821e1fe36cf1d7b6edaee913f3d57210319"
CR3_RESULT_SHA = "12b17fe5c3cc60574b9a95f7a69b8a6786a82cee456a6bb3682f9172b10110db"
CR3_STAGEA_COMMIT = "9711c5673c8d7eace445235aa00e2946d99cd443"
CR3_CODE = ["pork/cr3.py", "pork/cr3_reference.py"]
assert subprocess.run(["git", "diff", "--quiet", CR3_STAGEA_COMMIT, head, "--"] + CR3_CODE).returncode == 0, \
    "CR-3 のコードが CR-3 のときと違います。止めます。"
assert X.verify_lock(LOCK, (REPO / "notes/cr3_lock_stageA.json").read_text(),
                     (D3 / "STAGEA/CR3_stageA_lock.json").read_text()) == CR3_LOCK_SHA, "CR-3 の段A の記録が違います。止めます。"
_sp = D3 / "STAGEB/CR3_STAGEB_summary.json"
assert hashlib.sha256(_sp.read_bytes()).hexdigest() == CR3_STAGEB_SUMMARY_SHA, "CR-3 の段B の記録が違います。止めます。"
assert hashlib.sha256((D3 / "STAGEC/CR3_result.json").read_bytes()).hexdigest() == CR3_RESULT_SHA, "CR-3 の結果が違います。止めます。"
STB3 = json.loads((D3 / "STAGEB/CR3_STAGEB_state.json").read_text())
assert STB3["header"]["code_head"] == CR3_STAGEA_COMMIT and STB3["header"]["cr3_lock_sha256"] == CR3_LOCK_SHA
Z.use_engine_settings()                                   # CR-4: B_stop 10, chunk 500 (= CR-3's), 4 in flight
REQ3 = X.Requests3(ITEMS, TARGETS, K_F, LOCK["manifest_sha256"])
G.verify_collected(STB3, REQ3, D3 / "STAGEB")
RES3 = G.per_item_results(STB3, REQ3, D3 / "STAGEB")
QW = Z.add_window(X.word_quantities(TARGETS, ITEMS, TEXTS, RES3, K_F), items2)
_au = X.boot_items(X.item_auroc(*X._q1rows(QW["item"], QW["g_F"], QW["lab2"])))["value"]
assert abs(_au - Z.CR3_Q1_AUROC) < 1e-12, "CR-3 の 264 語の量を作り直した値が CR-3 の結果と違います。止めます（計画 E8）。"
NW = Z.build_nw(items2, TARGETS)
CELLS1 = Z.b1_cells(NW)
K1 = Z.b1_K(CELLS1)
ROWS1, M1 = Z.manifest_rows4(ITEMS, CELLS1, K1, {})
BUD1 = Z.budget_b1(len(K1), MAXTOK)
REFS = {"cr3_lock_sha256": CR3_LOCK_SHA, "cr3_stageB_summary_sha256": CR3_STAGEB_SUMMARY_SHA, "cr3_result_sha256": CR3_RESULT_SHA}
LOCKA = Z.lock_a(NW, CELLS1, BUD1, M1, len(ROWS1), REFS)
D4 = Path("/content/drive/MyDrive/DECISION_POINTS/cr4")
print("CR-2・CR-3 の照合: OK | NW", len(NW), "| 段B1 の切れ目", len(K1), "| リクエスト", len(ROWS1))

OUTA = D4 / "STAGEA"
OUTA.mkdir(parents=True, exist_ok=True)
(OUTA / "CR4_stageA_lock.json").write_text(Z.canonical(LOCKA))
(OUTA / "CR4_B1_manifest_PRIVATE.json").write_text(json.dumps({"manifest_sha256": M1, "rows": ROWS1}))
EXP = Z.explore(QW, TARGETS, items2, TEXTS, RES3)
exp_sha = write_json(OUTA / "CR4_explore_result.json", json.loads(Z.result_json(EXP)))
print("段A:", LOCKA["counts"], "| 予算の式 $%.2f" % BUD1["lhs_usd"], "| lock sha256:", Z.lock_sha256(LOCKA))
print("探索の結果 sha256:", exp_sha, "|", round(time.time() - t0, 1), "秒")
notify('終わりました：CR-4 の段A（一覧の固定と探索）を Drive に書きました（Claude に固定してもらってね）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()